# 04. Comparación de modelos de segmentación del ventrículo lateral

Todos los modelos se evalúan contra el experto (`gt == 4`) en los mismos estudios.

| Modelo | Tipo | Fuente |
|---|---|---|
| SynthSeg | red preentrenada (randomización de dominio) | incluido en HyKid |
| SynthSeg + ChP | SynthSeg + modelo de plexo coroideo | incluido en HyKid |
| SPM12 | modelo probabilístico de tejidos (GM/WM/LCR) | incluido en HyKid |
| Clásico | umbral + morfología + watershed (notebook 03) | `src/clasico.py` |
| FastSurfer, nnU-Net, BabySeg... | redes externas | opcional: poner predicciones en `outputs/predicciones/<modelo>/` |

**Métricas**: Dice, error de volumen (%), distancia de Hausdorff 95 (mm), por rango de volumen.

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(ROOT))

import shutil
import time
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi

from src.io import list_scans, load, load_seg, read_labels
from src.paths import OUTPUTS
import src.clasico as clasico

FIG_DIR = OUTPUTS / "figuras_04"
if FIG_DIR.exists():
    shutil.rmtree(FIG_DIR)
FIG_DIR.mkdir(parents=True)
PRED_DIR = OUTPUTS / "predicciones"

def guardar(nombre, dpi=110):
    plt.savefig(FIG_DIR / f"{nombre}.png", dpi=dpi, bbox_inches="tight")

scans = list_scans()
print(len(scans), "estudios")

## Definición de modelos

Cada modelo es una función `(scan, img, zooms) -> máscara binaria del ventrículo lateral` o `None` si no está disponible.

- **SynthSeg / SynthSeg+ChP / FastSurfer**: convención FreeSurfer, ventrículo lateral = 4, 43; inferior = 5, 44.
- **SPM12**: da LCR total, no ventrículo. Se toma el LCR (probabilidad > 0.5) y se conservan las componentes que no tocan el borde del cerebro. Se reporta para mostrar por qué un mapa de tejidos no basta.
- **Externos**: archivo `outputs/predicciones/<modelo>/<sub>_<scan>.nii.gz` en el espacio de `eT1W_FFE_SVR`, con la etiqueta del ventrículo en `EXTERNOS`.

In [ ]:
FS_VENT = [4, 5, 43, 44]
SPM_CSF_CANAL = 2
SPM_UMBRAL = 0.5

EXTERNOS = {
    "FastSurfer": [4, 5, 43, 44],
    "nnUNet_ventriculos": [1],
    "BabySeg": [4, 5, 43, 44],
    "UNet3D": [1],
}


def leer_seg(path):
    if not path.exists():
        return None
    return np.asarray(nib.as_closest_canonical(nib.load(path)).dataobj)


def m_synthseg(s, img, zooms):
    a = leer_seg(s / "segmentation" / "synthseg.nii.gz")
    return None if a is None else np.isin(a.astype(int), FS_VENT)


def m_synthseg_chp(s, img, zooms):
    a = leer_seg(s / "segmentation" / "synthseg+ChP.nii.gz")
    return None if a is None else np.isin(a.astype(int), FS_VENT)


def m_spm12(s, img, zooms):
    a = leer_seg(s / "segmentation" / "spm12.nii.gz")
    if a is None:
        return None
    if a.ndim == 4:
        csf = a[..., SPM_CSF_CANAL] > SPM_UMBRAL
    elif np.issubdtype(a.dtype, np.floating) and a.max() <= 1.0:
        csf = a > SPM_UMBRAL
    else:
        csf = a.astype(int) == 3
    brain = clasico.brain_mask(img)
    csf &= brain
    rim = brain & ~ndi.binary_erosion(brain, clasico.ball, iterations=3)
    lab, n = ndi.label(csf)
    if n == 0:
        return csf
    tocan = set(np.unique(lab[rim])) - {0}
    return np.isin(lab, [i for i in range(1, n + 1) if i not in tocan])


def m_clasico(s, img, zooms):
    v, *_ = clasico.segmentar(img, zooms)
    return v


def m_externo(nombre, etiquetas):
    def f(s, img, zooms):
        a = leer_seg(PRED_DIR / nombre / f"{s.parent.name}_{s.name}.nii.gz")
        return None if a is None else np.isin(a.astype(int), etiquetas)
    return f


MODELOS = {"SynthSeg": m_synthseg, "SynthSeg+ChP": m_synthseg_chp, "SPM12": m_spm12, "Clásico": m_clasico}
for nombre, et in EXTERNOS.items():
    if (PRED_DIR / nombre).exists():
        MODELOS[nombre] = m_externo(nombre, et)
print("Modelos:", list(MODELOS))

## Inspección de formatos

In [ ]:
s0 = scans[0]
print("Etiquetas experto:", read_labels())
for f in ["gt", "synthseg", "synthseg+ChP", "spm12"]:
    p = s0 / "segmentation" / f"{f}.nii.gz"
    if p.exists():
        a = np.asarray(nib.load(p).dataobj)
        u = np.unique(a)
        print(f"{f:14s} shape={a.shape} dtype={a.dtype} valores={u[:12]}{' ...' if len(u) > 12 else ''}")

## Métricas

In [ ]:
def dice(a, b):
    d = a.sum() + b.sum()
    return 2 * (a & b).sum() / d if d else np.nan


def hd95(a, b, zooms):
    if a.sum() == 0 or b.sum() == 0:
        return np.nan
    ba = a & ~ndi.binary_erosion(a)
    bb = b & ~ndi.binary_erosion(b)
    da = ndi.distance_transform_edt(~bb, sampling=zooms)[ba]
    db = ndi.distance_transform_edt(~ba, sampling=zooms)[bb]
    return np.percentile(np.concatenate([da, db]), 95)

## Evaluación en todos los estudios

In [ ]:
filas, guardadas = [], {}
for s in scans:
    t0 = time.time()
    img, zooms = load(s / "eT1W_FFE_SVR.nii.gz")
    gt, _ = load_seg(s / "segmentation" / "gt.nii.gz")
    g = gt == 4
    ml = float(np.prod(zooms)) / 1000
    box = clasico.crop_box(img)
    fila = {"subject": s.parent.name, "scan": s.name, "vol_gt": g.sum() * ml}
    masks = {}
    for nombre, f in MODELOS.items():
        try:
            if nombre in ("Clásico", "SPM12"):
                m = np.zeros_like(g)
                sub = f(s, img[box], zooms)
                if sub is None:
                    continue
                m[box] = sub
            else:
                m = f(s, img, zooms)
            if m is None or m.shape != g.shape:
                continue
        except Exception as e:
            print(f"  {nombre} falló en {s.parent.name}: {e}")
            continue
        masks[nombre] = m
        fila[f"dice_{nombre}"] = dice(m, g)
        fila[f"vol_{nombre}"] = m.sum() * ml
        fila[f"err_{nombre}_%"] = (m.sum() - g.sum()) / g.sum() * 100 if g.sum() else np.nan
        fila[f"hd95_{nombre}"] = hd95(m, g, zooms)
    filas.append(fila)
    guardadas[(s.parent.name, s.name)] = (s, masks)
    print(f"{s.parent.name}/{s.name} ({time.time() - t0:.0f}s): " +
          " | ".join(f"{n} {fila.get(f'dice_{n}', np.nan):.2f}" for n in MODELOS))

df = pd.DataFrame(filas)
df["rango"] = pd.cut(df.vol_gt, [0, 30, 150, np.inf], labels=["pequeño (<30)", "mediano (30-150)", "grande (>150)"])
df.to_csv(FIG_DIR / "comparacion_todos.csv", index=False)

## Resumen por modelo y rango

In [ ]:
nombres = [n for n in MODELOS if f"dice_{n}" in df]
rows = []
for grupo, sub in [("total", df)] + list(df.groupby("rango", observed=True)):
    for n in nombres:
        rows.append({"rango": grupo, "modelo": n, "n": sub[f"dice_{n}"].notna().sum(),
                     "dice_media": sub[f"dice_{n}"].mean(), "dice_mediana": sub[f"dice_{n}"].median(),
                     "dice_std": sub[f"dice_{n}"].std(),
                     "err_vol_abs_mediana_%": sub[f"err_{n}_%"].abs().median(),
                     "hd95_mediana_mm": sub[f"hd95_{n}"].median()})
resumen = pd.DataFrame(rows)
resumen.to_csv(FIG_DIR / "resumen_modelos.csv", index=False)
display(resumen.round(3))

tabla = resumen.pivot(index="modelo", columns="rango", values="dice_mediana").round(3)
display(tabla)

## Gráficos

In [ ]:
colores = dict(zip(nombres, plt.cm.tab10.colors))
fig, ax = plt.subplots(1, 3, figsize=(19, 5))

rangos = list(df.rango.cat.categories)
w = 0.8 / len(nombres)
for i, n in enumerate(nombres):
    pos = np.arange(len(rangos)) + (i - (len(nombres) - 1) / 2) * w
    datos = [df.loc[df.rango == r, f"dice_{n}"].dropna() for r in rangos]
    bp = ax[0].boxplot(datos, positions=pos, widths=w * 0.9, patch_artist=True, showfliers=False)
    for b in bp["boxes"]:
        b.set_facecolor(colores[n])
ax[0].set_xticks(range(len(rangos)))
ax[0].set_xticklabels([f"{r}\nn={(df.rango == r).sum()}" for r in rangos])
ax[0].set_ylabel("Dice"); ax[0].set_ylim(0, 1); ax[0].set_title("Dice por rango de volumen")
ax[0].legend([plt.Rectangle((0, 0), 1, 1, color=colores[n]) for n in nombres], nombres, fontsize=8)

for n in nombres:
    ax[1].scatter(df.vol_gt, df[f"dice_{n}"], s=18, color=colores[n], label=n)
ax[1].set_xscale("log"); ax[1].set_xlabel("volumen experto (mL, log)"); ax[1].set_ylabel("Dice")
ax[1].set_ylim(0, 1); ax[1].set_title("Dice según severidad"); ax[1].legend(fontsize=8)

lim = [1, df[[f"vol_{n}" for n in nombres] + ["vol_gt"]].max().max() * 1.2]
for n in nombres:
    ax[2].scatter(df.vol_gt, df[f"vol_{n}"], s=18, color=colores[n], label=n)
ax[2].plot(lim, lim, "k--", lw=1)
ax[2].set_xscale("log"); ax[2].set_yscale("log")
ax[2].set_xlabel("volumen experto (mL)"); ax[2].set_ylabel("volumen predicho (mL)")
ax[2].set_title("Concordancia de volumen"); ax[2].legend(fontsize=8)
plt.tight_layout(); guardar("comparacion_resumen"); plt.show()

## Bland-Altman del volumen

In [ ]:
fig, axes = plt.subplots(1, len(nombres), figsize=(4.5 * len(nombres), 4), squeeze=False)
for ax, n in zip(axes[0], nombres):
    m = (df[f"vol_{n}"] + df.vol_gt) / 2
    d = df[f"vol_{n}"] - df.vol_gt
    sesgo, sd = d.mean(), d.std()
    ax.scatter(m, d, s=15, color=colores[n])
    for y, ls in [(sesgo, "-"), (sesgo + 1.96 * sd, "--"), (sesgo - 1.96 * sd, "--")]:
        ax.axhline(y, color="k", ls=ls, lw=1)
    ax.set_title(f"{n}\nsesgo {sesgo:+.1f} mL, LoA ±{1.96 * sd:.0f}", fontsize=9)
    ax.set_xlabel("media (mL)"); ax.set_ylabel("predicho - experto (mL)")
plt.tight_layout(); guardar("bland_altman"); plt.show()

## Visualización: 6 casos del más pequeño al más grande

In [ ]:
orden = df.sort_values("vol_gt")
picks = orden.iloc[np.linspace(0, len(orden) - 1, min(6, len(orden))).astype(int)]
fig, axes = plt.subplots(len(picks), len(nombres) + 1, figsize=(3.3 * (len(nombres) + 1), 3.3 * len(picks)), squeeze=False)
for row, (_, r) in zip(axes, picks.iterrows()):
    s, masks = guardadas[(r.subject, r.scan)]
    img, _ = load(s / "eT1W_FFE_SVR.nii.gz")
    g = load_seg(s / "segmentation" / "gt.nii.gz")[0] == 4
    k = int(np.argmax(g.sum(axis=(0, 1))))
    sl = lambda a: np.rot90(a[:, :, k])
    paneles = [(g, f"{r.subject} experto {r.vol_gt:.0f} mL", "purple")]
    paneles += [(masks.get(n), f"{n} {r.get(f'dice_{n}', np.nan):.2f}", colores[n]) for n in nombres]
    for ax, (m, t, c) in zip(row, paneles):
        ax.imshow(sl(img), cmap="gray")
        if m is not None:
            ax.contourf(sl(m), levels=[0.5, 1.5], colors=[c], alpha=0.45)
            if t[0] != "s":
                ax.contour(sl(g), levels=[0.5], colors="white", linewidths=0.6)
        ax.set_title(t, fontsize=9)
        ax.axis("off")
plt.tight_layout(); guardar("comparacion_visual"); plt.show()
print("Archivos en", FIG_DIR)

## Cómo agregar más modelos

1. Correr el modelo (FastSurfer, nnU-Net, BabySeg, etc.) sobre `eT1W_FFE_SVR.nii.gz` de cada estudio (ej. en Google Colab con GPU).
2. Guardar cada salida como `outputs/predicciones/<modelo>/<sub_XXX>_<scan_XX>.nii.gz` en el mismo espacio y tamaño.
3. Ajustar la etiqueta del ventrículo en `EXTERNOS` y volver a correr. El notebook lo detecta solo.